# 🔢 NB-14: Multi-label Skill Classification
**Goal:** Classify each Claude response with multiple skill tags (reasoning, coding, math, creativity, science).

**Modality:** Multi-label Classification | **Model:** DistilBERT

In [ ]:
!pip install transformers datasets scikit-learn -q

In [ ]:
import json, re, random
random.seed(42)

def load_dataset(path="claude-opus-4_5-250x_jsonl.txt"):
    """Load Claude thinking dataset from JSONL."""
    records = []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records

def extract_fields(records):
    data = []
    for r in records:
        msgs = r["messages"]
        user = next((m["content"] for m in msgs if m["role"]=="user"), "")
        asst = next((m["content"] for m in msgs if m["role"]=="assistant"), "")
        think = re.findall(r"<think>(.*?)</think>", asst, re.DOTALL)
        response = re.sub(r"<think>.*?</think>", "", asst, flags=re.DOTALL).strip()
        data.append({
            "user": user,
            "assistant_full": asst,
            "thinking": " ".join(think).strip(),
            "response": response
        })
    return data

records = load_dataset()
data = extract_fields(records)
print(f"Loaded {len(data)} examples")
print(f"Sample user: {data[0]['user'][:80]}")
print(f"Sample think: {data[0]['thinking'][:120]}...")
print(f"Sample resp: {data[0]['response'][:120]}...")


In [ ]:
import numpy as np, torch, torch.nn as nn
from transformers import DistilBertTokenizer, DistilBertModel, TrainingArguments, Trainer
from torch.utils.data import Dataset as TDS
from sklearn.metrics import f1_score

SKILLS = ["coding","math","science","philosophy","reasoning","creativity","writing","analysis"]

def multi_label(text):
    text = text.lower()
    return [
        int(any(k in text for k in ["code","python","function","class","algorithm"])),
        int(any(k in text for k in ["equation","calculate","theorem","proof","integral"])),
        int(any(k in text for k in ["physics","biology","chemistry","quantum","molecule"])),
        int(any(k in text for k in ["consciousness","ethics","meaning","existence"])),
        int(any(k in text for k in ["step","therefore","because","thus","reason","logic"])),
        int(any(k in text for k in ["imagine","creative","story","metaphor","art"])),
        int(any(k in text for k in ["write","essay","explain","describe","discuss"])),
        int(any(k in text for k in ["analyze","compare","evaluate","assess","examine"])),
    ]

labels = [multi_label(d["response"]) for d in data]
texts  = [f"{d['user']} [SEP] {d['response'][:300]}" for d in data]

tok = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")
enc = tok(texts, truncation=True, padding="max_length", max_length=256, return_tensors="pt")

class MLDS(TDS):
    def __len__(self): return len(labels)
    def __getitem__(self, i):
        return {**{k: v[i] for k,v in enc.items()}, "labels": torch.tensor(labels[i], dtype=torch.float)}

class MultiLabelBERT(nn.Module):
    def __init__(self):
        super().__init__()
        self.bert = DistilBertModel.from_pretrained("distilbert-base-uncased")
        self.head = nn.Linear(768, len(SKILLS))
    def forward(self, input_ids, attention_mask, labels=None):
        cls = self.bert(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state[:,0]
        logits = self.head(cls)
        loss = nn.BCEWithLogitsLoss()(logits, labels) if labels is not None else None
        from transformers.modeling_outputs import SequenceClassifierOutput
        return SequenceClassifierOutput(loss=loss, logits=logits)

model = MultiLabelBERT()
n = len(labels); split = int(n*0.85)
full = MLDS()
train_ds = torch.utils.data.Subset(full, range(split))
eval_ds  = torch.utils.data.Subset(full, range(split, n))

def metrics(p):
    preds = (torch.sigmoid(torch.tensor(p.predictions)) > 0.5).int().numpy()
    return {"f1_macro": f1_score(p.label_ids, preds, average="macro", zero_division=0)}

args = TrainingArguments("./multilabel-claude", num_train_epochs=5,
    per_device_train_batch_size=16, fp16=torch.cuda.is_available(),
    evaluation_strategy="epoch", report_to="none")
Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=eval_ds,
        compute_metrics=metrics).train()
print("✅ Multi-label classifier ready!")
